# M11.2 · MDP, değer yinelemesi ve yakınsama sınırları

Markov karar süreci: durumlar, eylemler, geçiş modeli, ödül, indirim, ufuk. Sonlu ufukta geriye doğru dinamik programlama ve zamana bağlı (durağan olmayan) en iyi politika; sonsuz ufukta Bellman optimallik operatörünün γ-büzülme olduğu, sabit noktası ve "n adım sonra ne kadar uzaktayım?" sınırı. Notlardaki perakende stok örneğini elle ve saf Python'da tam çözüyorsun; NumPy'da einsum'lu Bellman operatörü, durma kuralı ve sınırın rastgele MDP'lerde sağlandığını; PyTorch'ta yığın halinde yüzlerce MDP ve autograd ile $\partial V^*/\partial R$'nin indirimli ziyaret ölçüsü olduğunu; en sonda büyük bir stok MDP'sini vektörel çözüyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-2-mdp-vi.html](https://m-gonul.github.io/pytorch-lab/m11-2-mdp-vi.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_2_mdp_vi.py
CHECKS = {
 "b": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef ref(P, R, VT, h, g):\n    P, R = np.array(P), np.array(R)\n    V = np.array(VT, float); Vs = [V]; pis = []\n    for _ in range(h):\n        Q = R + g * np.einsum(\"ast,t->sa\", P, V)\n        pis.append(Q.argmax(1)); V = Q.max(1); Vs.append(V)\n    return Vs[::-1], pis[::-1]\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def ret():\n        P, R, VT = need(ns, \"retail_mdp\")()\n        ensure(np.allclose(np.array(P).sum(-1), 1), \"satırlar olasılık\")\n        Vs, pis = need(ns, \"finite_vi\")(P, R, VT, 5)\n        Ws, qis = ref(P, R, VT, 5, 1.0)\n        for t in range(6):\n            close(np.array(Vs[t]), Ws[t], atol=1e-9, what=f\"V_{t}\")\n        for t in range(5):\n            ensure(list(pis[t])[1:] == list(qis[t])[1:], f\"π_{t}\")\n        close(np.array(Vs[4]), np.array([-9, -1.8, -5.8, -10.8]), atol=1e-9, what=\"V_4 (notlar)\")\n        name = lambda a: \"yüksek\" if a == 0 else \"indirim\"\n        R_.table(\"Perakende stok, h = 5: en iyi eylem (S1, S2, S3)\", [\"t (kalan gün)\", \"S1\", \"S2\", \"S3\", \"V_t(S1)\"],\n                 [[f\"{t} ({5 - t})\", name(pis[t][1]), name(pis[t][2]), name(pis[t][3]), f\"{Vs[t][1]:.4f}\"] for t in range(5)],\n                 note=\"Son günde her stok düzeyinde indirim; daha fazla gün kalınca S1 ve sonra S2'de yüksek fiyata geçiş: durağan olmayan politika.\")\n    R_.test(\"perakende problemi (finite_vi)\", ret)\n\n    def rnd():\n        rng = np.random.default_rng(0)\n        P = rng.random((3, 6, 6)); P /= P.sum(-1, keepdims=True); R = rng.normal(size=(6, 3)); VT = rng.normal(size=6)\n        Vs, pis = ns[\"finite_vi\"](P.tolist(), R.tolist(), VT.tolist(), 7, 0.9)\n        Ws, qis = ref(P, R, VT, 7, 0.9)\n        for t in range(8):\n            close(np.array(Vs[t]), Ws[t], atol=1e-9, what=f\"rastgele MDP V_{t}\")\n    R_.test(\"rastgele MDP\", rnd)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef rand_mdp(rng, S=8, A=3):\n    P = rng.random((A, S, S)) ** 3; P /= P.sum(-1, keepdims=True)\n    return P, rng.normal(size=(S, A))\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"yalnız yineleme döngüsü\", lambda: max_loops(src, 1, func=\"value_iteration\"))\n    rng = np.random.default_rng(0)\n\n    def pe():\n        P, R = rand_mdp(rng)\n        pi = rng.integers(0, 3, 8)\n        V = need(ns, \"policy_eval\")(P, R, pi, 0.9)\n        Tp = P[pi, np.arange(8)]; Rp = R[np.arange(8), pi]\n        close(V, Rp + 0.9 * Tp @ V, atol=1e-10, what=\"V = R_π + γT_πV\")\n    R_.test(\"policy_eval (doğrusal sistem)\", pe)\n\n    def bd():\n        rows, worst = [], 0\n        for g in (0.5, 0.9, 0.99):\n            for _ in range(10):\n                P, R = rand_mdp(rng)\n                Vstar, pistar, _, _ = need(ns, \"value_iteration\")(P, R, g, 1e-12)\n                for eps in (1e-1, 1e-3):\n                    V, pi, n, d = ns[\"value_iteration\"](P, R, g, eps)\n                    err = np.abs(Vstar - V).max()\n                    ensure(err <= need(ns, \"bound\")(g, d) + 1e-9, f\"γ = {g}: hata {err:.2e} > sınır {ns['bound'](g, d):.2e}\")\n                    loss = np.abs(Vstar - ns[\"policy_eval\"](P, R, pi, g)).max()\n                    ensure(loss <= 2 * g * eps / (1 - g) + 1e-9, \"politika kaybı sınırı\")\n                    worst = max(worst, err / ns[\"bound\"](g, d))\n            P, R = rand_mdp(rng)\n            _, _, n, d = ns[\"value_iteration\"](P, R, g, 1e-6)\n            rows.append([g, n, f\"{d:.1e}\", f\"{ns['bound'](g, d):.1e}\"])\n        R_.table(\"Değer yinelemesi, ε = 1e-6\", [\"γ\", \"iterasyon\", \"son fark\", \"V*'a uzaklık sınırı\"], rows,\n                 note=f\"Gerçek hata / sınır oranının en büyüğü {worst:.2f}: sınır geçerli ve oldukça sıkı. γ → 1'de iterasyon ~ log ε / log γ ile patlar.\")\n    R_.test(\"durma sınırı ve politika kaybı\", bd)\n",
 "d": "import math\nimport torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    g = torch.Generator().manual_seed(0)\n    B, A, S = 300, 3, 10\n    P = torch.rand(B, A, S, S, generator=g, dtype=torch.float64) ** 3; P /= P.sum(-1, keepdim=True)\n    R = torch.randn(B, S, A, generator=g, dtype=torch.float64)\n\n    def bv():\n        V, d = need(ns, \"batched_vi\")(P, R, 0.9, 200)\n        ensure(tuple(V.shape) == (B, S) and tuple(d.shape) == (200, B), \"şekiller\")\n        Q = R + 0.9 * torch.einsum(\"bast,bt->bsa\", P, V)\n        close(V.numpy(), Q.max(-1).values.numpy(), atol=1e-8, what=\"sabit nokta\")\n        ratio = (d[1:] / d[:-1].clamp_min(1e-300))[:60]\n        ensure(bool((ratio <= 0.9 + 1e-9).all()), \"ardışık farklar her adımda en az γ katına inmeli\")\n        need_n = [int((d[:, b] > 1e-6).sum()) + 1 for b in range(B)]\n        theory = math.ceil(math.log(1e-6 / d[0].max().item()) / math.log(0.9))\n        return f\"1e-6'ya kadar iterasyon: en az {min(need_n)}, en çok {max(need_n)}; teorik üst sınır log(ε/‖V₁−V₀‖)/log γ ≈ {theory}\"\n    R_.test(\"batched_vi (300 MDP)\", bv)\n\n    def cr():\n        r = need(ns, \"contraction_ratio\")(P[0], R[0], 0.9)\n        ensure(float(r.max()) <= 0.9 + 1e-12, f\"oran {float(r.max()):.4f} > γ\")\n        return f\"en büyük oran {float(r.max()):.4f} (γ = 0.9), ortalama {float(r.mean()):.3f}\"\n    R_.test(\"contraction_ratio ≤ γ\", cr)\n\n    def grad():\n        Pm, Rm = P[1], R[1].clone().requires_grad_(True)\n        V = need(ns, \"soft_v_star\")(Pm, Rm, 0.9, 400)\n        V[0].backward()\n        with torch.no_grad():\n            pi = (Rm + 0.9 * torch.einsum(\"ast,t->sa\", Pm, V)).argmax(-1)\n            Tp = Pm[pi, torch.arange(S)]\n            d = torch.linalg.solve(torch.eye(S, dtype=torch.float64) - 0.9 * Tp.T, torch.eye(S, dtype=torch.float64)[0])\n            occ = torch.zeros(S, A, dtype=torch.float64); occ[torch.arange(S), pi] = d\n        close(Rm.grad.numpy(), occ.numpy(), atol=1e-8, what=\"∂V*(s0)/∂R = indirimli ziyaret ölçüsü\")\n        close(float(Rm.grad.sum()), 1 / (1 - 0.9), atol=1e-8, what=\"toplam 1/(1 − γ)\")\n    R_.test(\"autograd: ∂V*/∂R = d^π*\", grad)\n",
 "e": "import numpy as np\nfrom scipy.stats import poisson\nfrom labcheck import close, need, ensure, bench, fmt_time\n\n\ndef check(ns, R_):\n    def mdp():\n        P, R = need(ns, \"inventory_mdp\")(C=20, max_order=8, lam=4.0)\n        ensure(P.shape == (9, 21, 21) and R.shape == (21, 9), f\"şekiller {P.shape}, {R.shape}\")\n        close(P.sum(-1), np.ones((9, 21)), atol=1e-12, what=\"satırlar olasılık\")\n        s, a = 5, 3\n        y = 8; d = np.arange(29); pd = poisson.pmf(d, 4.0); pd[-1] += poisson.sf(28, 4.0)\n        want = (pd * (10 * np.minimum(y, d) - 0.5 * np.maximum(y - d, 0) - 2 * np.maximum(d - y, 0))).sum() - 4 * a\n        close(R[s, a], want, atol=1e-10, what=\"beklenen ödül\")\n        close(P[a, s, 2], pd[6], atol=1e-12, what=\"geçiş: 8 − 6 = 2\")\n    R_.test(\"inventory_mdp\", mdp)\n\n    def solve():\n        P, R = ns[\"inventory_mdp\"]()\n        V, pi = need(ns, \"vi_vectorized\")(P, R, 0.95)\n        close(V, ns[\"vi_loop\"](P, R, 0.95), atol=1e-6, what=\"döngülü VI ile aynı\")\n        post = np.minimum(np.arange(51) + pi, 50)\n        low = np.where(pi > 0)[0]\n        ensure(len(low) > 0 and (post[low] == post[low][0]).mean() > 0.8, \"sipariş verilen durumlarda hedef düzey sabit olmalı\")\n        ensure((pi[low.max() + 1:] == 0).all(), \"eşiğin üstünde sipariş yok\")\n        t1 = bench(lambda: ns[\"vi_loop\"](P, R, 0.95, 1e-6), repeat=1, number=1)\n        t2 = bench(lambda: ns[\"vi_vectorized\"](P, R, 0.95, 1e-6), repeat=3)\n        R_.table(\"Stok MDP (51 durum × 21 eylem), γ = 0.95\", [\"ölçü\", \"değer\"], [[\"sipariş eşiği s\", int(low.max())], [\"hedef düzey S\", int(post[low][0])], [\"döngülü VI\", fmt_time(t1)], [\"vektörel VI\", fmt_time(t2)]],\n                 note=\"(s, S) politikası: stok s'nin altına inince S'ye tamamla. Scarf (1960) bu yapının sabit sipariş maliyetinde en iyi olduğunu kanıtladı; burada sabit maliyet yok, yine de eşik yapısı çıkıyor.\")\n    R_.test(\"vi_vectorized ve (s, S) yapısı\", solve)\n",
 "x1": "import numpy as np\nfrom labcheck import close, need, ensure\n\nACT = [\"oturma odası\", \"mutfak\", \"şarj\", \"bekle\"]\n\n\ndef check(ns, R_):\n    def t():\n        P, R = need(ns, \"vacuum_mdp\")()\n        ensure(P.shape == (4, 8, 8) and R.shape == (8, 4), \"şekiller\")\n        close(P.sum(-1), np.ones((4, 8)), atol=1e-12, what=\"satırlar olasılık\")\n        close(R[1, 0], 0.9 * 10 - 1 + 0 - 0, atol=1e-9, what=\"kirli oturma odası, pil yüksek, temizle: 0.9·10 − 1\")\n        close(R[1 + 4, 0], 0.5 * (-51) + 0.5 * (0.9 * 10 - 1), atol=1e-9, what=\"pil düşükken temizleme riski\")\n        V, pi = need(ns, \"solve\")(0.95)\n        ensure(all(pi[s] == 2 for s in range(4, 8) if s != 4), \"pil düşük ve kirli oda varken önce şarj\")\n        rows = [[s & 1, (s >> 1) & 1, \"düşük\" if s >> 2 else \"yüksek\", ACT[pi[s]], f\"{V[s]:.1f}\"] for s in range(8)]\n        R_.table(\"Robot süpürge, γ = 0.95\", [\"oturma kirli\", \"mutfak kirli\", \"pil\", \"en iyi eylem\", \"V\"], rows)\n    R_.test(\"robot süpürge\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef retail():\n    P = np.zeros((2, 4, 4))\n    for a, sale in enumerate((0.3, 0.8)):\n        P[a, 0, 1] = 1\n        for k in range(1, 4):\n            P[a, k, k - 1] = sale; P[a, k, k] = 1 - sale\n    R = np.array([[-5, -5], [-1, -1], [-1, -1], [-2, -2]], float)\n    return P, R, np.array([0, -4, -8, -12.])\n\n\ndef check(ns, R_):\n    P, R, VT = retail()\n    name = lambda p: \"\".join(\"Y\" if a == 0 else \"İ\" for a in p[1:])\n\n    def h():\n        pols, k = need(ns, \"stationary_policy_horizon\")(P, R, VT, 1.0, 15)\n        ensure(len(pols) == 15 and name(pols[0]) == \"İİİ\" and name(pols[1]) == \"Yİİ\", \"h = 1 ve 2'deki politikalar\")\n        R_.table(\"Perakende: başlangıç politikası (S1 S2 S3; Y yüksek, İ indirim)\", [\"h\", \"politika\"], [[i + 1, name(p)] for i, p in enumerate(pols) if i < 6 or i == 14],\n                 note=f\"h ≥ {k} için politika değişmiyor: uzun ufukta durağan politikaya yaklaşılır.\")\n    R_.test(\"ufukla politika\", h)\n\n    def g():\n        out = need(ns, \"policy_vs_gamma\")(P, R, [0.1, 0.5, 0.8, 0.95, 0.99])\n        ensure(len(out) == 5, \"5 politika\")\n        R_.table(\"Sonsuz ufuk (terminal yok): γ'ya göre politika\", [\"γ\", \"politika (S1 S2 S3)\"], [[gg, name(p)] for gg, p in zip([0.1, 0.5, 0.8, 0.95, 0.99], out)],\n                 note=\"Küçük γ: yalnız anlık maliyete bakılır (tüm eylemler aynı maliyette → eşitlikte ilk eylem); büyük γ: stoksuz kalma cezası planlamaya girer.\")\n    R_.test(\"γ ile politika\", g)\n",
 "x3": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        r, g = need(ns, \"find_l2_counterexample\")(20000, 0)\n        ensure(r > g, f\"L2 oranı {r:.3f}, γ = {g:.3f}: γ'yı aşan bir örnek bulunmalı\")\n        rng = np.random.default_rng(1)\n        worst = 0\n        for _ in range(5000):\n            P = rng.random((2, 3, 3)) ** 4; P /= P.sum(-1, keepdims=True)\n            R = rng.normal(size=(3, 2)); gg = rng.uniform(0.5, 0.99)\n            V, U = rng.normal(size=3) * 5, rng.normal(size=3) * 5\n            worst = max(worst, np.abs(need(ns, \"h_star\")(P, R, gg, V) - ns[\"h_star\"](P, R, gg, U)).max() / np.abs(V - U).max() / gg)\n        ensure(worst <= 1 + 1e-12, \"sonsuz normda oran γ'yı aşmamalı\")\n        return f\"L2'de oran {r:.3f} > γ = {g:.3f} (büzülme değil); ∞-normda en büyük oran/γ = {worst:.4f} ≤ 1\"\n    R_.test(\"büzülme normun seçimine bağlı\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_2_mdp_vi as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**MDP:** durumlar $S=\{S_0,…,S_M\}$, eylemler $A$, geçiş modeli $P(s'|s,a)$, ödül $R(s,a)$, indirim $0\le\gamma\le1$, ufuk $h$. **Politika** $\pi(s)=a$ (tam gözlenebilir durum). Varsayımlar: tam gözlem, bilinen ve durağan model, ayrık zaman/durum/eylem. Amaç: beklenen indirimli ödül toplamını $\sum_t\gamma^tR(s_t,a_t)$ en büyük yapan politika. İndirim hem sonsuz toplamı sonlu kılar hem de erken ödülleri tercih eder.

**Değer fonksiyonu (sonlu ufuk):** $V_h(s)=R_h(s)$ (son adım ödülü genelde farklıdır: kazandı/kaybetti, elde kalan stok…) ve geriye doğru

$$V_t(s)=\max_a\Big[R(s,a)+\gamma\sum_{s'}P(s'|s,a)V_{t+1}(s')\Big].$$

Bu **değer yinelemesi** (dinamik programlama) her adım için ayrı bir eylem seçer: en iyi politika **durağan değildir** (kalan süreye bağlı).

**Sonsuz ufuk:** vektör biçiminde $V_{n}=\max_a(R^a+\gamma T^aV_{n-1})$ ($\max$ eleman bazında, $T^a$ satır-stokastik). Sabit nokta $V^*=H^*V^*$; açgözlü politika $\pi^*(s)=\arg\max_a[R(s,a)+\gamma\sum P V^*]$ **durağandır**.

**Politika değerlendirme:** sabit $\pi$ için $V=R_\pi+\gamma T_\pi V$ doğrusal: $V^\pi=(I-\gamma T_\pi)^{-1}R_\pi$ (ya da yinele).

**Yakınsama:** $H(V)=R+\gamma TV$ ve $H^*(V)=\max_a(R^a+\gamma T^aV)$ sonsuz normda **γ-büzülmedir**. Banach: tek sabit nokta, her başlangıçtan yakınsama, $\lVert H^{(n)}V-V^*\rVert_\infty\le\gamma^n\lVert V-V^*\rVert_\infty$. Pratik **durma sınırı**: $\lVert V_n-V_{n-1}\rVert_\infty\le\varepsilon$ ise $\lVert V^*-V_n\rVert_\infty\le\frac{\gamma}{1-\gamma}\varepsilon$.

## 1 · Elle

Perakende örneği: $S_0$ stok yok ($-5$), $S_1$ az ($-1$), $S_2$ orta ($-1$), $S_3$ çok ($-2$). Yüksek fiyat: %30 satış (bir stok aşağı), %70 değişmez; indirim: %80 / %20. $S_0$ gece yeniden stoklanır: $S_1$'e geçer.

$V_4(S_1)$: yüksek $-1+0.3\cdot0+0.7\cdot(-4)=-3.8$, indirim $-1+0.8\cdot0+0.2\cdot(-4)=-1.8$ → indirim. Bir gün daha kalınca $V_3(S_1)$: yüksek $-1+0.3(-9)+0.7(-1.8)=-4.96$, indirim $-1+0.8(-9)+0.2(-1.8)=-8.56$ → **yüksek fiyat**: stoksuz kalmanın $-5$'lik cezası, bir gün daha varken satışı yavaşlatmayı değerli kılar.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Perakende stok (notlar), $h=5$, $\gamma=1$: son gün değerleri $V_5=[0,-4,-8,-12]$**

- `v4s0`: $V_4(S_0)=R(S_0)+V_5(S_1)$
- `v4s1`: $V_4(S_1)$ (en iyi eylemle)
- `v4s3`: $V_4(S_3)$
- `v3s1`: $V_3(S_1)$
- `a3s1`: $t=3$'te $S_1$ için en iyi eylem (yüksek fiyat 1, indirim 2)

**Sınırlar, $\gamma=0.9$**

- `bd`: $\lVert V_n-V_{n-1}\rVert_\infty=0.01$ ise $\lVert V^*-V_n\rVert_\infty$ üst sınırı
- `geo`: Her adımda ödül 1 (sonsuz ufuk): $V$

In [ ]:
cevaplar = dict(
    v4s0=None,
    v4s1=None,
    v4s3=None,
    v3s1=None,
    a3s1=None,
    bd=None,
    geo=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"v4s0":[-9,0],"v4s1":[-1.8,0],"v4s3":[-10.8,0],"v3s1":[-4.96,0],"a3s1":[1,0],"bd":[0.09,0],"geo":[10,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, genel MDP: `P[a][s][s2]`, `R[s][a]`.

- `backup(P, R, V, gamma, s)` — tüm eylemler için $Q(s,a)=R(s,a)+\gamma\sum_{s'}P(s'|s,a)V(s')$ listesi.
- `finite_vi(P, R, VT, h, gamma=1.0)` → `(Vs, pis)`: `Vs[t]` $t=0..h$ ($V_h=VT$), `pis[t]` $t=0..h-1$ en iyi eylem indisleri (eşitlikte küçük indis).
- `retail_mdp()` → `(P, R, VT)`: notlardaki perakende problemi; eylem 0 yüksek fiyat, 1 indirim; $S_0$'da iki eylem de $S_1$'e götürür.

In [ ]:
%%writefile stage_b.py
def backup(P, R, V, gamma, s):
    ...


def finite_vi(P, R, VT, h, gamma=1.0):
    ...


def retail_mdp():
    ...


P, R, VT = retail_mdp()
Vs, pis = finite_vi(P, R, VT, 5)
print(Vs[4], pis[3])

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, `P (A, S, S)` satır-stokastik, `R (S, A)`. Yalnız yineleme döngüsü serbest.

- `q_values(P, R, V, gamma)` — `(S, A)`, `einsum("ast,t->sa", P, V)`.
- `value_iteration(P, R, gamma, eps=1e-8, V0=None)` → `(V, pi, n, last_diff)`: durma kuralı $\lVert V_n-V_{n-1}\rVert_\infty\le\varepsilon$.
- `policy_eval(P, R, pi, gamma)` — $(I-\gamma T_\pi)^{-1}R_\pi$ (`np.linalg.solve`), $T_\pi$ = `P[pi, arange(S)]`.
- `bound(gamma, diff)` — $\frac{\gamma}{1-\gamma}\,$`diff`.

Kontrol, rastgele MDP'lerde (1) sınırın sağlandığını, (2) çıkarılan politikanın kaybının $\frac{2\gamma\varepsilon}{1-\gamma}$'yı aşmadığını sınar.

In [ ]:
%%writefile stage_c.py
import numpy as np


def q_values(P, R, V, gamma):
    ...


def value_iteration(P, R, gamma, eps=1e-8, V0=None):
    ...


def policy_eval(P, R, pi, gamma):
    ...


def bound(gamma, diff):
    ...


rng = np.random.default_rng(0)
P = rng.random((2, 4, 4)); P /= P.sum(-1, keepdims=True)
print(value_iteration(P, rng.random((4, 2)), 0.9)[2])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `batched_vi(P, R, gamma, iters)` — `P (B, A, S, S)`, `R (B, S, A)`: B MDP aynı anda; `iters` adım; `(V (B, S), diffs (iters, B))` (her adımın sonsuz norm farkı).
- `contraction_ratio(P, R, gamma, n=1000, seed=0)` — rastgele $V,U$ çiftleri için $\lVert H^*V-H^*U\rVert_\infty/\lVert V-U\rVert_\infty$ (tek MDP, `(n,)` tensör).
- `soft_v_star(P, R, gamma, iters)` — `R` `requires_grad` ile türevlenebilir VI (aynı Bellman, `max` ile): autograd ile $\partial V^*(s_0)/\partial R(s,a)$ = **indirimli ziyaret ölçüsü** $d^{\pi^*}(s,a)=\sum_t\gamma^tP(s_t=s,a_t=a\,|\,s_0)$. Kontrol bunu kapalı formla karşılaştırır.

In [ ]:
%%writefile stage_d.py
import torch


def batched_vi(P, R, gamma, iters):
    ...


def contraction_ratio(P, R, gamma, n=1000, seed=0):
    ...


def soft_v_star(P, R, gamma, iters):
    ...


P = torch.rand(4, 2, 5, 5, dtype=torch.float64); P /= P.sum(-1, keepdim=True)
print(batched_vi(P, torch.rand(4, 5, 2, dtype=torch.float64), 0.9, 50)[0].shape)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Büyük bir stok MDP'si, vektörel.

- `inventory_mdp(C=50, max_order=20, lam=8.0, price=10.0, cost=4.0, hold=0.5, short=2.0)` → `(P, R)`: durum stok $0..C$, eylem sipariş $0..$`max_order` (stoğu $C$'de kırp), talep $D\sim$ Poisson($\lambda$) (kuyruğu $C+$`max_order`'da birikir). Satış $\min(s+a,D)$; ödül $=$ fiyat·beklenen satış − maliyet·$a$ − tutma·beklenen kalan − kıtlık·beklenen karşılanmayan. **Döngüsüz** (`scipy.stats.poisson.pmf`, yayın).
- `vi_vectorized(P, R, gamma, eps)` ve karşılaştırma için durum döngülü `vi_loop` (verildi).

Kontrol, en iyi politikanın "(s, S) politikası" yapısını (stok bir eşiğin altındayken hedef düzeye tamamla) gösterir.

In [ ]:
%%writefile stage_e.py
import numpy as np
import torch
from scipy.stats import poisson


def inventory_mdp(C=50, max_order=20, lam=8.0, price=10.0, cost=4.0, hold=0.5, short=2.0):
    ...


def vi_vectorized(P, R, gamma, eps=1e-8):
    ...


def vi_loop(P, R, gamma, eps=1e-8):
    A, S, _ = P.shape
    V = np.zeros(S)
    while True:
        Vn = np.empty(S)
        for s in range(S):
            Vn[s] = max(R[s, a] + gamma * P[a, s] @ V for a in range(A))
        if np.abs(Vn - V).max() <= eps:
            return Vn
        V = Vn


print("stok")

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Robot süpürge MDP'sini kur ve çöz

Notlardaki robot süpürge (2 oda × temiz/kirli × pil yüksek/düşük = 8 durum; durum indisi `living + 2·kitchen + 4·low`, 1 = kirli/düşük).

Eylemler: 0 oturma odasını temizle, 1 mutfağı temizle, 2 şarj et, 3 bekle (oda değiştirme, temizleme eylemine dahil). Dinamik:

- Temizleme: oda kirliyse 0.9 olasılıkla temizlenir (+10), her durumda −1 pil maliyeti; pil yüksekse 0.3 olasılıkla düşer; pil düşükse 0.5 olasılıkla **tükenir**: −50 ve pil yüksek olarak sıfırlanır (kurtarma), odalar değişmez.
- Şarj: pil yüksek olur, 0 ödül. Bekle: 0 ödül.
- Her adımda her oda 0.1 olasılıkla yeniden kirlenir (eylemden sonra). İki oda da temizse (eylemden önce) +20.

`vacuum_mdp()` → `(P (4, 8, 8), R (8, 4))`, `solve(gamma=0.95)` → `(V, pi)`.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def vacuum_mdp():
    ...


def solve(gamma=0.95):
    ...


print(solve()[1])

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Ufuk ve indirim: politika ne zaman değişir?

- `stationary_policy_horizon(P, R, VT, gamma, H)` — perakende problemi için $h=1..H$: başlangıçtaki (t = 0) politika listesi; politikanın değişmeyi bıraktığı en küçük $h$.
- `policy_vs_gamma(P, R, gammas)` — sonsuz ufuk (terminal yok) için her γ'da en iyi politika.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def stationary_policy_horizon(P, R, VT, gamma, H):
    ...


def policy_vs_gamma(P, R, gammas):
    ...


print("ufuk")

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Büzülme yalnız sonsuz normda: L2'de karşı örnek

- `h_star(P, R, gamma, V)` — Bellman optimallik operatörü (tek MDP).
- `find_l2_counterexample(trials=20000, seed=0)` — rastgele küçük MDP'ler ve $V,U$ çiftleri arasında $\lVert H^*V-H^*U\rVert_2/\lVert V-U\rVert_2$ oranının en büyüğünü ve ilgili $\gamma$'yı döndür: $\gamma$'yı aşan bir örnek bulunmalı (büzülme norma bağlıdır). Sonsuz normda oran asla $\gamma$'yı geçmez.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def h_star(P, R, gamma, V):
    ...


def find_l2_counterexample(trials=20000, seed=0):
    ...


print(find_l2_counterexample(200))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Value Iteration Networks

[Tamar, Wu, Thomas, Levine, Abbeel — Value Iteration Networks (NeurIPS 2016)](https://arxiv.org/abs/1602.02867)

Değer yinelemesini bir CNN katmanı olarak yaz: ödül haritası ve değer haritası kanalları üzerinde 3 × 3 konvolüsyon ($\\bar Q$), kanal boyunca max ($V$), K kez tekrar; ızgara dünyasında harita görüntüsünden ödülü öğrenen ve VI modülünü uçtan uca eğiten VIN'i uygula. Rastgele engel haritalarında yol bulma başarısını sıradan bir CNN politika ağıyla, eğitimde görülmeyen büyük haritalarda karşılaştır.


**Adımlar**

1. Veri: rastgele engelli 16 × 16 ve 28 × 28 haritalar, hedef ve en kısa yol eylemleri (BFS).
2. VI modülü: $\bar Q=W_r*R+W_v*V$, $V=\max_a\bar Q$; K = 20 (16 × 16), K = 36 (28 × 28).
3. Dikkat (attention): ajanın konumundaki $\bar Q$ vektörünü seç, eylem sınıflandırması.
4. Bu modülün d aşamasındaki türevlenebilir VI ile bağlantıyı tartış.

**Kabul ölçütleri**

- 28 × 28'de VIN başarı oranı ≥ %85, düz CNN'den en az 15 puan yüksek.
- K'ya karşı başarı grafiği; K < harita çapı olduğunda başarının düştüğü gösterilmiş.

## Kaynaklar

- [Sutton & Barto — Reinforcement Learning: An Introduction (2. baskı), Bölüm 3–4](http://incompleteideas.net/book/the-book-2nd.html) — CC BY-NC-ND 2.0
- [Puterman — Markov Decision Processes (Wiley 1994), Bölüm 4 ve 6](https://doi.org/10.1002/9780470316887) — ©
- [Poupart — CS885 Reinforcement Learning (Waterloo), Ders 2–3](https://cs.uwaterloo.ca/~ppoupart/teaching/cs885-spring20/) — doğrulanmadı
- [Bertsekas — Dynamic Programming and Optimal Control, Cilt 2](http://www.athenasc.com/dpbook.html) — ©